In [1]:
import pandas as pd
import numpy as np
import pyreadr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon

warnings.simplefilter(action="ignore", category=pd.errors.SettingWithCopyWarning)

NameError: name 'warnings' is not defined

In [ ]:
s1 = pd.read_excel('/home/lucytian/data/9_PIP_comparison/Data_S1.xlsx', skiprows=1)

### Credible set size

In [ ]:
group_cols = ["Trait", "Merged Signal", "Population"]
signal_summary = (
    s1.groupby(group_cols, dropna=False)
       .agg(
           n_variants=("RSID", "size"),
           top_pip=("Overall PIP", "max"),
           cs_size=("CS-Level Pip", lambda x: x.notna().sum())
       )
       .reset_index()
)

In [ ]:
ss = signal_summary.copy()
ss = ss[ss["Population"].isin(["EUR", "AFR"])]

counts = (
    ss.groupby(["Trait", "Merged Signal"])["Population"]
      .nunique()
      .reset_index(name="n_pop")
)

matched = counts[counts["n_pop"] == 2][["Trait", "Merged Signal"]]

ss = ss.merge(matched, on=["Trait", "Merged Signal"], how="inner")

In [ ]:
paired = ss.pivot_table(
    index=["Trait", "Merged Signal"],
    columns="Population",
    values=["top_pip", "cs_size"]
)

# flatten columns
paired.columns = [f"{m}_{p}" for m, p in paired.columns]
paired = paired.reset_index()

In [ ]:
paired["delta_cs"] = paired["cs_size_AFR"] - paired["cs_size_EUR"]
paired["delta_pip"] = paired["top_pip_AFR"] - paired["top_pip_EUR"]

paired["afr_smaller_cs"] = paired["cs_size_AFR"] < paired["cs_size_EUR"]
paired["afr_smaller_pip"] = paired["top_pip_AFR"] < paired["top_pip_EUR"]

In [ ]:
from scipy.stats import wilcoxon

# CS size: AFR < EUR
w_cs = wilcoxon(paired["cs_size_AFR"], paired["cs_size_EUR"], alternative="less")

# PIP: AFR < EUR
w_pip = wilcoxon(paired["top_pip_AFR"], paired["top_pip_EUR"], alternative="less")

print(w_cs)
print(w_pip)

In [ ]:
print("Median ΔCS:", paired["delta_cs"].median())
print("% AFR smaller CS:", paired["afr_smaller_cs"].mean())

print("Median ΔPIP:", paired["delta_pip"].median())
print("% AFR smaller PIP:", paired["afr_smaller_pip"].mean())

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(paired["cs_size_EUR"], paired["cs_size_AFR"], alpha=0.3, color='purple')
mx = max(paired["cs_size_EUR"].max(), paired["cs_size_AFR"].max())
plt.plot([0, mx], [0, mx], '--', color='purple')
plt.xlabel("EUR CS size")
plt.ylabel("AFR CS size")
plt.title("Credible set size: EUR vs AFR")
#plt.savefig('credible_set_size.pdf', bbox_inches='tight')
plt.show()

### PIP cumulative distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6, 4))

# -----------------
# 1. AFR CDF Curve
# -----------------
afr_data = paired["top_pip_AFR"].dropna()
x_sorted_afr = np.sort(afr_data)
y_cdf_afr = np.arange(1, len(x_sorted_afr) + 1) / len(x_sorted_afr)
ax.plot(x_sorted_afr, y_cdf_afr, color='tab:red', linewidth=1.5, label='AFR PIP')

# -----------------
# 2. EUR CDF Curve
# -----------------
eur_data = paired["top_pip_EUR"].dropna()
x_sorted_eur = np.sort(eur_data)
y_cdf_eur = np.arange(1, len(x_sorted_eur) + 1) / len(x_sorted_eur)
ax.plot(x_sorted_eur, y_cdf_eur, color='tab:blue', linewidth=1.5, label='EUR PIP')

# -----------------
# Formatting & Legend
# -----------------
ax.set_xlabel("PIP value")
ax.set_ylabel("Cumulative Probability")
ax.set_ylim(0, 1)
ax.legend(loc='upper left', frameon=True)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
#plt.savefig('pip_cdf_combined.pdf', bbox_inches='tight')
plt.show()

### Log Odds Ratio across cV2F decile

In [ ]:
s2 = s1[s1['Population'].isin(['AFR', 'EUR'])]
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# --- 1. Identify which Loci are "successful" ---
# We find the max PIP within each Locus/Population combo
locus_max_pip = s2.groupby(['Locus', 'Population'])['Overall PIP'].max().unstack()

# Create sets of Loci names based on your 90% threshold
eur_success = set(locus_max_pip[locus_max_pip['EUR'] >= 0.9].index)
afr_success = set(locus_max_pip[locus_max_pip['AFR'] >= 0.9].index)

# --- 2. Create the 4 Group Definitions ---
both = eur_success.intersection(afr_success)
only_eur = eur_success - afr_success
only_afr = afr_success - eur_success
# Everything else is "Neither"

# --- 3. Assign these labels back to EVERY variant in the main DF ---
def get_group(locus):
    if locus in both: return "Both (AFR & EUR)"
    if locus in only_eur: return "EUR Only"
    if locus in only_afr: return "AFR Only"
    return "Neither"

s2['Locus_Classification'] = s2['Locus'].apply(get_group)

In [ ]:
base_p = '~/group/data/ENCODE4/cV2F/20231221/scores/mvp_eur/baseline_annotation_chrmbpnet/'
dfs = []
for i in np.arange(1, 23):
    df = pd.read_csv(base_p + 'mvp_eur_all_baseline_annotation_chrmbpnet/mvp_eur_0.9_0.01_all_baseline_annotation_chrmbpnet_lava_ld_ld_maf.' + str(i) + '.cv2f.txt.gz', sep='\t', compression='gzip')
    dfs.append(df)
    
eur_df = pd.concat(dfs)

In [ ]:
base_p = '~/group/data/ENCODE4/cV2F/20231221/scores/mvp_afr/baseline_annotation_chrmbpnet/'
dfs = []
for i in np.arange(1, 23):
    df = pd.read_csv(base_p + 'mvp_afr_all_baseline_annotation_chrmbpnet/mvp_afr_0.9_0.01_all_baseline_annotation_chrmbpnet_lava_ld_ld_maf.' + str(i) + '.cv2f.txt.gz', sep='\t', compression='gzip')
    dfs.append(df)
    
afr_df = pd.concat(dfs)

In [ ]:
merged = s2.merge(eur_df, left_on='RSID', right_on='SNP')
merged = merged.rename(columns={'cV2F': 'eur_cV2F'})
merged = merged.drop(columns='SNP')
merged = merged.merge(afr_df, left_on='RSID', right_on='SNP')
merged = merged.rename(columns={'cV2F': 'afr_cV2F'})
both = merged[merged['Locus_Classification'] == "Both (AFR & EUR)"]
both['cV2F'] = np.where(both['Population'] == 'EUR', 
                      both['eur_cV2F'], 
                      both['afr_cV2F'])
afr_pip = both[both['Population'] == 'AFR']
afr_pip['high_pip'] = (afr_pip['Overall PIP'] >= 0.9).astype(int)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pseudo = 0.5

def compute_log_or(df, bin_col, pseudo=0.5):
    grouped = df.groupby(bin_col)

    summary = grouped['high_pip'].agg(['sum', 'count']).rename(
        columns={'sum': 'high', 'count': 'total'}
    )

    summary['low'] = summary['total'] - summary['high']

    # global totals
    total_high = df['high_pip'].sum()
    total_low = len(df) - total_high

    # define contingency table
    summary['a'] = summary['high']
    summary['b'] = summary['low']
    summary['c'] = total_high - summary['a']
    summary['d'] = total_low - summary['b']

    # log odds ratio
    summary['log_or'] = np.log(
        (summary['a'] + pseudo) * (summary['d'] + pseudo) /
        ((summary['b'] + pseudo) * (summary['c'] + pseudo))
    )

    # standard error
    summary['se'] = np.sqrt(
        1/(summary['a'] + pseudo) +
        1/(summary['b'] + pseudo) +
        1/(summary['c'] + pseudo) +
        1/(summary['d'] + pseudo)
    )

    return summary

# Example: create deciles separately
afr_pip['afr_cV2F_bin'] = pd.qcut(afr_pip['afr_cV2F'], 10, labels=False, duplicates='drop')
afr_pip['eur_cV2F_bin'] = pd.qcut(afr_pip['eur_cV2F'], 10, labels=False, duplicates='drop')

# Compute summaries
afr_curve = compute_log_or(afr_pip, 'afr_cV2F_bin')
eur_curve = compute_log_or(afr_pip, 'eur_cV2F_bin')

# Plot overlay
plt.figure(figsize=(7,5))

plt.errorbar(
    afr_curve.index,
    afr_curve['log_or'],
    yerr=afr_curve['se'],
    fmt='o-',
    capsize=4,
    color='tab:red',
    label='AFR cV2F'
)

plt.errorbar(
    eur_curve.index,
    eur_curve['log_or'],
    yerr=eur_curve['se'],
    fmt='o-',
    capsize=4,
    color='tab:blue',
    label='EUR cV2F'
)

plt.xlabel("cV2F decile")
plt.ylabel("Log Odds Ratio (PIP ≥ 0.9)")
plt.axhline(0, linestyle='--')
plt.legend()
#plt.savefig('decile.pdf', bbox_inches='tight')
plt.show()